# 04 — Analysis and Visualization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = '/content/drive/MyDrive/BSE405_SoilMoisture/results'
FIG_DIR = '/content/drive/MyDrive/BSE405_SoilMoisture/figures'

import os
os.makedirs(FIG_DIR, exist_ok=True)

results = pd.read_csv(f'{RESULTS_DIR}/model_results.csv')
predictions = pd.read_csv(f'{RESULTS_DIR}/predictions_spatial.csv')
importance = pd.read_csv(f'{RESULTS_DIR}/feature_importance.csv')

print(results.to_string(index=False))

## Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pivot_rmse = results.pivot(index='model', columns='split', values='test_rmse')
pivot_rmse.plot(kind='bar', ax=axes[0], rot=15)
axes[0].set_title('Test RMSE by Model and Split Strategy')
axes[0].set_ylabel('RMSE (m³/m³)')
axes[0].legend(title='Split')

pivot_r2 = results.pivot(index='model', columns='split', values='test_r2')
pivot_r2.plot(kind='bar', ax=axes[1], rot=15)
axes[1].set_title('Test R² by Model and Split Strategy')
axes[1].set_ylabel('R²')
axes[1].legend(title='Split')

plt.suptitle('Model Comparison Across Split Strategies', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Overfitting Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

random_results = results[results['split'] == 'Random 80/20']

x = np.arange(len(random_results))
width = 0.35

ax.bar(x - width/2, random_results['train_r2'], width, label='Train R²', color='steelblue')
ax.bar(x + width/2, random_results['test_r2'], width, label='Test R²', color='coral')

ax.set_xlabel('Model')
ax.set_ylabel('R²')
ax.set_title('Train vs Test R² (Random Split) — Overfitting Analysis')
ax.set_xticks(x)
ax.set_xticklabels(random_results['model'], rotation=15)
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/overfitting_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Importance

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='teal')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Spatial Error Map

In [ ]:
best_model = results.loc[results[results['split'] == 'West->East']['test_rmse'].idxmin(), 'model']
print(f"Best model (West->East): {best_model}")

pred_col = f'pred_{best_model}'
predictions['error'] = predictions[pred_col] - predictions['actual']

plt.figure(figsize=(10, 8))
scatter = plt.scatter(predictions['longitude'], predictions['latitude'],
                      c=predictions['error'], cmap='RdBu_r', s=3, alpha=0.5,
                      vmin=-0.2, vmax=0.2)
plt.colorbar(scatter, label='Prediction Error (m³/m³)')
plt.xlabel('Longitude (°)')
plt.ylabel('Latitude (°)')
plt.title(f'Spatial Error Map — {best_model} (Train: West, Test: East)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/spatial_error_map.png', dpi=150, bbox_inches='tight')
plt.show()

## Residual Distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(predictions['error'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(x=0, color='red', linestyle='--', label='Zero error')
plt.xlabel('Prediction Error (m³/m³)')
plt.ylabel('Frequency')
plt.title(f'Residual Distribution — {best_model} (West→East)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/residual_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean error (bias): {predictions['error'].mean():.4f}")
print(f"Std of error: {predictions['error'].std():.4f}")

## Spatial Generalization Analysis

In [ ]:
split_order = ['Random 80/20', 'West->East', 'Spatial Block CV']

fig, ax = plt.subplots(figsize=(10, 6))

for model_name in results['model'].unique():
    model_data = results[results['model'] == model_name]
    model_data = model_data.set_index('split').loc[split_order]
    ax.plot(split_order, model_data['test_r2'], marker='o', label=model_name)

ax.set_xlabel('Split Strategy')
ax.set_ylabel('Test R²')
ax.set_title('Spatial Generalization — R² Degradation Across Split Strategies')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/spatial_degradation.png', dpi=150, bbox_inches='tight')
plt.show()